[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/17_feature_engineering/01_numeric_transforms.ipynb)

# Bài 01 — Biến đổi feature số

**🎯 Mục tiêu**
- Thấy sức mạnh của feature engineering: **cùng một model**, feature tốt hơn → kết quả tốt hơn hẳn.
- Xử lý phân phối lệch: log, Box-Cox / Yeo-Johnson, quantile transform.
- Biểu diễn quan hệ phi tuyến cho model tuyến tính: binning, **spline**.
- Feature miền (domain): tỉ số, hiệu, đơn vị có ý nghĩa; winsorize outlier.
- Model nào cần biến đổi nào (tuyến tính vs cây).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/17_feature_engineering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (FunctionTransformer, KBinsDiscretizer, OneHotEncoder, PowerTransformer, QuantileTransformer,
                                   SplineTransformer, StandardScaler)

from ml_course.datasets import make_house_prices
from ml_course.visualization import set_style

set_style()
houses = make_house_prices(n=3000, seed=4)
y = houses["price_billion_vnd"]
houses.head()

## 1. Cùng model, khác feature

Linear Regression trên feature thô vs trên feature được biến đổi theo hiểu biết về bài toán: giá nhà **tỉ lệ** với diện tích và hệ số của quận (quan hệ **nhân**), nên log-log tuyến tính hơn.

In [ ]:
def cv_mae(model, X, y, log_target=False):
    if log_target:
        model = TransformedTargetRegressor(model, func=np.log, inverse_func=np.exp)
    return -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error").mean()


num = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
raw = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(), ["district"])])
engineered = ColumnTransformer([
    ("log_area", make_pipeline(FunctionTransformer(np.log, feature_names_out="one-to-one"), StandardScaler()), ["area_m2"]),
    ("num", StandardScaler(), ["bedrooms", "bathrooms", "age_years", "distance_to_center_km"]),
    ("garden", "passthrough", ["has_garden"]),
    ("cat", OneHotEncoder(), ["district"]),
])
X = houses.drop(columns=["price_billion_vnd"]).assign(has_garden=lambda d: d.has_garden.astype(float))
rows = [("Linear, feature thô", cv_mae(make_pipeline(raw, LinearRegression()), X, y)),
        ("Linear, log(area) + log(giá)", cv_mae(make_pipeline(engineered, LinearRegression()), X, y, log_target=True)),
        ("HistGB, feature thô", cv_mae(make_pipeline(raw, HistGradientBoostingRegressor(random_state=0)), X, y))]
for name, mae in rows:
    print(f"{name:30s} MAE (5-fold) = {mae:.3f} tỉ")

Một phép biến đổi đúng (log hai vế → quan hệ nhân thành cộng) đưa model **tuyến tính** ngang hoặc hơn model phức tạp — mà vẫn giữ được tính đơn giản, dễ diễn giải. Feature engineering tốt thường đáng giá hơn tuning.

## 2. Phân phối lệch: log, Box-Cox, Yeo-Johnson, Quantile

Feature/target lệch phải (thu nhập, giá, số lượt xem) gây khó cho model tuyến tính và các phương pháp dựa trên khoảng cách: vài giá trị cực lớn chi phối.

| Biến đổi | Điều kiện | Ghi chú |
|---|---|---|
| $\log(x)$, $\log(1 + x)$ | $x > 0$ / $x \ge 0$ | đơn giản, diễn giải được (% thay đổi) |
| Box-Cox $\frac{x^\lambda - 1}{\lambda}$ | $x > 0$ | học $\lambda$ để gần Gaussian nhất |
| Yeo-Johnson | mọi $x$ | mở rộng Box-Cox cho số âm/0 |
| Quantile → uniform/normal | mọi $x$ | ép phân phối bất kỳ; mất khoảng cách tương đối; robust với outlier |

In [ ]:
rng = np.random.default_rng(0)
income = rng.lognormal(np.log(15), 0.9, 3000)
transforms = {"gốc": income, "log": np.log(income),
              "Box-Cox": PowerTransformer(method="box-cox").fit_transform(income[:, None]).ravel(),
              "Quantile → normal": QuantileTransformer(output_distribution="normal", n_quantiles=500).fit_transform(income[:, None]).ravel()}
fig, axes = plt.subplots(1, 4, figsize=(20, 3.5))
for ax, (name, v) in zip(axes, transforms.items()):
    ax.hist(v, bins=60)
    ax.set_title(f"{name}: skew = {pd.Series(v).skew():.2f}")
plt.show()

## 3. Quan hệ phi tuyến cho model tuyến tính: binning và spline

Giá nhà theo tuổi nhà có thể không đơn điệu (nhà mới đắt, nhà rất cũ ở khu trung tâm cũng đắt...). Model tuyến tính chỉ vẽ được **một đường thẳng** theo mỗi feature — trừ khi ta mở rộng feature:
- **Binning**: chia thành khoảng → hằng số từng khúc (giống cây).
- **Spline**: đa thức từng khúc, nối mượt tại các nút — linh hoạt và ổn định hơn polynomial bậc cao.

In [ ]:
x = np.sort(rng.uniform(0, 10, 400))
y_nl = np.sin(x) + 0.3 * x + rng.normal(0, 0.3, 400)
grid = np.linspace(0, 10, 300).reshape(-1, 1)
models = {"tuyến tính": LinearRegression(),
          "binning (10 bin) + linear": make_pipeline(KBinsDiscretizer(10, encode="onehot-dense", strategy="uniform"), LinearRegression()),
          "spline (8 nút, bậc 3) + Ridge": make_pipeline(SplineTransformer(n_knots=8, degree=3), Ridge(alpha=1e-3))}
plt.scatter(x, y_nl, s=8, alpha=0.4, color="gray")
for name, m in models.items():
    m.fit(x[:, None], y_nl)
    cv = -cross_val_score(m, x[:, None], y_nl, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_squared_error").mean()
    plt.plot(grid, m.predict(grid), lw=2, label=f"{name}: MSE CV = {cv:.3f}")
plt.legend(fontsize=8)
plt.title("Mở rộng feature cho phép model tuyến tính học quan hệ phi tuyến")
plt.show()

## 4. Feature miền: tỉ số, hiệu, đơn vị có ý nghĩa

Model tuyến tính và cây đều **khó** tự học tỉ số $a/b$ (cây cần rất nhiều split). Kiến thức miền giúp tạo feature mạnh:

In [ ]:
X_dom = X.assign(area_per_bedroom=X.area_m2 / X.bedrooms,
                 bath_per_bed=X.bathrooms / X.bedrooms,
                 log_area=np.log(X.area_m2),
                 is_old=(X.age_years > 30).astype(float))
base_cols = num + ["has_garden"]
dom_cols = base_cols + ["area_per_bedroom", "bath_per_bed", "log_area", "is_old"]
for name, cols in [("feature gốc", base_cols), ("+ feature miền", dom_cols)]:
    pre = ColumnTransformer([("num", StandardScaler(), cols), ("cat", OneHotEncoder(), ["district"])])
    print(f"{name:15s} Linear (log giá): MAE = {cv_mae(make_pipeline(pre, LinearRegression()), X_dom, y, log_target=True):.3f}")

## 5. Winsorize: chặn outlier

Một vài giá trị cực đoan (lỗi nhập liệu, khách hàng "cá voi") kéo lệch hệ số của model tuyến tính. **Winsorize** = chặn ở phân vị (vd 1% và 99%) — tính phân vị **trên train** (trong Pipeline):

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class Winsorizer(TransformerMixin, BaseEstimator):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_, self.hi_ = np.quantile(X, self.lower, axis=0), np.quantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)


dirty = make_house_prices(n=3000, seed=4, dirty=True).dropna().drop_duplicates()
dirty["district"] = dirty["district"].str.strip().str.replace("Q.", "Quận", regex=False).str.title()
Xd, yd = dirty.drop(columns=["price_billion_vnd"]).assign(has_garden=lambda d: d.has_garden.astype(float)), dirty["price_billion_vnd"]
for name, num_pipe in [("không winsorize", StandardScaler()), ("winsorize 1%–99%", make_pipeline(Winsorizer(), StandardScaler()))]:
    pre = ColumnTransformer([("num", num_pipe, num), ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"])])
    print(f"{name:18s} Linear: MAE = {cv_mae(make_pipeline(pre, LinearRegression()), Xd, yd):.3f}")

Dữ liệu "bẩn" có ~1% diện tích bị gõ thừa số 0 (×10) — winsorize giảm ảnh hưởng của chúng lên model tuyến tính. (Tốt hơn nữa: sửa lỗi dữ liệu tại nguồn — chương 04.)

## 6. Model nào cần biến đổi nào?

| Biến đổi | Linear / Logistic / SVM / KNN / Neural net | Cây, Random Forest, Gradient Boosting |
|---|---|---|
| Scaling | **cần** | không cần |
| Log / power transform | thường giúp nhiều | không đổi gì (bất biến biến đổi đơn điệu) |
| Binning / spline | giúp học phi tuyến | không cần |
| Tỉ số, hiệu, tương tác miền | giúp | **giúp** (cây khó tự học tỉ số) |
| Winsorize | giúp | ít tác dụng |

## 🧠 Kiểm tra nhanh

1. Vì sao log(giá) ~ log(diện tích) thường tuyến tính hơn giá ~ diện tích?
2. Box-Cox và Yeo-Johnson khác nhau ở đâu?
3. Spline có ưu điểm gì so với polynomial bậc cao?
4. Vì sao feature tỉ số giúp cả model cây?

## 📌 Tóm tắt

- Feature tốt > model phức tạp; log/power transform cho dữ liệu lệch và quan hệ nhân.
- Binning/spline cho model tuyến tính học phi tuyến; feature miền (tỉ số) giúp mọi model.
- Mọi biến đổi học từ dữ liệu (Box-Cox λ, phân vị) phải fit trên train — trong Pipeline.

## 📚 Tài liệu tham khảo

- Zheng & Casari (2018) — *Feature Engineering for Machine Learning*
- Kuhn & Johnson (2019) — *Feature Engineering and Selection* (bookdown.org/max/FES)
- scikit-learn User Guide — Preprocessing data